# Replay one experimental game

This notebook replays one Choi 2020 game and shows:

- the replay setup in separate stages
- the actions printed per step
- the directed graph after each recorded state
- replay metrics explored interactively with an observer selector


## Imports


In [ ]:
import sys
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import scienceplots  # noqa: F401
from IPython.display import display
from matplotlib.offsetbox import AnnotationBbox, DrawingArea
from matplotlib.patches import Circle, FancyArrowPatch
from matplotlib_inline.backend_inline import set_matplotlib_formats

project_root = Path.cwd()
if not (project_root / "network_creation").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from network_creation.core.observability import ObservationSampling
from network_creation.data.choi_2020 import (
    build_replay_simulator,
    list_choi_2020_configs,
    load_choi_2020_normalized_data,
    run_choi_2020_replay_batch,
)
from network_creation.data.choi_2020_references import (
    build_choi_observers,
    load_reference_parameters,
)
from network_creation.observers.grimoire import OBSERVER_GRIMOIRE
from network_creation.plotting.episode_series import plot_episode_series_observer
from network_creation.plotting.graph_history import (
    build_circle_layout,
    plot_graph_history,
)
from network_creation.plotting.reference_values import get_sigmoid_fit_reference_values
from notebooks.utils.plot_sigmoid_fit import plot_sigmoid_fit

ActionPrintObserver = OBSERVER_GRIMOIRE["action_print"]
StateHistoryObserver = OBSERVER_GRIMOIRE["state_history"]

In [ ]:
%matplotlib inline
plt.style.use("science")
set_matplotlib_formats("svg", bbox_inches=None)
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 17,
    "axes.labelsize": 15,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "legend.title_fontsize": 13,
    "figure.titlesize": 19,
})

In [ ]:
choi_parameters = {
    row.group_size: row
    for row in load_reference_parameters(project_root / "data/choi_2020/parameters.json")
}
metric_observer_names = (
    "density", "indegree", "global_closeness_centrality", "distance",
    "action_count", "hub_persistence", "central_point_of_dominance",
    "triad_census", "centered_mean_welfare", "agent_payoff"
)
metric_observer_settings = {"hub_persistence": {"window_size": 10}}
aggregate_observer_aggregations = {
    "*": ("mean", "std"),
    "density": ("mean", "std", "runs"),
    "central_point_of_dominance": ("mean", "std", "runs"),
    "centered_mean_welfare": ("mean", "std", "runs"),
    "hub_persistence": ("mean", "std", "runs"),
    "agent_payoff": ("mean", "std", "runs"),
}


def make_metric_observers(group_size, aggregations=()):
    parameters = choi_parameters[int(group_size)]
    return build_choi_observers(
        n_agents=int(group_size),
        observer_names=metric_observer_names,
        observation_sampling=ObservationSampling.periodic(1),
        aggregations=aggregations,
        benefit_to_cost_ratio=parameters.benefit_to_cost_ratio,
        distance_discount=parameters.distance_discount,
        settings_by_observer=metric_observer_settings,
    )


AGENT_PAYOFF_PLOT_DIMENSIONS = (
    "highest_degree",
    "second_highest_degree",
    "rest_except_top_two",
)
AGENT_PAYOFF_ROLE_LABELS = {
    "highest_degree": "top 1 degree",
    "second_highest_degree": "top 2 degree",
    "rest_except_top_two": "rest mean",
}
AGENT_PAYOFF_ROLE_COLORS = {
    "highest_degree": "#1f4e79",
    "second_highest_degree": "#c84c09",
    "rest_except_top_two": "#3a7d44",
}


def observer_plot_dimensions(name, observer):
    if name == "agent_payoff":
        return (None,)
    return getattr(observer, "dimension_names", None) or (None,)


def observer_plot_specs(observers_by_name):
    return [
        (name, observer, dimension)
        for name, observer in observers_by_name.items()
        for dimension in observer_plot_dimensions(name, observer)
    ]


def observer_plot_label(observer, dimension_name):
    if dimension_name is None:
        return observer.label
    return f"{observer.label} - {dimension_name}"


def display_observer_explorer(observers_by_name, build_plot):
    options = [
        (observer_plot_label(observer, dimension_name), (name, dimension_name))
        for name, observer, dimension_name in observer_plot_specs(observers_by_name)
    ]
    control = widgets.Dropdown(
        description="Observer",
        options=options,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="360px"),
    )
    controls = widgets.HBox(
        [control],
        layout=widgets.Layout(
            gap="12px",
            align_items="flex-start",
            flex_flow="row wrap",
        ),
    )

    def current_plot():
        figure = build_plot(control.value)
        plt.close(figure)
        return figure

    def update_plot(_):
        try:
            plot_display.update(current_plot())
        except ValueError as error:
            plot_display.update(str(error))

    display(controls)
    plot_display = display(current_plot(), display_id=True)
    control.observe(update_plot, names="value")
    return control, plot_display


TRIAD_GRAPHLET_NODE_POSITIONS = {
    "a": (8, 8),
    "b": (80, 8),
    "c": (44, 56),
}


def set_observer_plot_title(axis, observer_name, observer, dimension_name):
    axis.set_title(observer_plot_label(observer, dimension_name))


def add_triad_graphlet(axis, dimension_name, *, below_legend):
    drawing = DrawingArea(88, 64, 0, 0)
    triad_graph = nx.triad_graph(dimension_name)
    for source, target in triad_graph.edges():
        drawing.add_artist(
            FancyArrowPatch(
                TRIAD_GRAPHLET_NODE_POSITIONS[source],
                TRIAD_GRAPHLET_NODE_POSITIONS[target],
                arrowstyle="-|>",
                mutation_scale=10,
                linewidth=1.5,
                color="black",
                shrinkA=6,
                shrinkB=6,
            )
        )
    for position in TRIAD_GRAPHLET_NODE_POSITIONS.values():
        drawing.add_artist(
            Circle(
                position,
                4.5,
                facecolor="0.75",
                edgecolor="black",
                linewidth=0.9,
            )
        )

    graphlet = AnnotationBbox(
        drawing,
        (1.13, 0.32 if below_legend else 0.68),
        xycoords=axis.transAxes,
        frameon=True,
        pad=0,
        bboxprops={
            "boxstyle": "round,pad=0.35",
            "facecolor": "white",
            "edgecolor": "0.35",
            "linewidth": 0.8,
        },
    )
    graphlet.set_in_layout(False)
    axis.add_artist(graphlet)


def move_axis_legend_outside(axis):
    legend = axis.get_legend()
    if legend is None:
        return False
    legend_title = legend.get_title().get_text()
    axis.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
        title=legend_title or None,
    )
    return True


def finalize_metric_figure(figure, axes, observer_name, dimension_name):
    plot_axes = np.atleast_1d(axes).ravel()
    legend_flags = [move_axis_legend_outside(axis) for axis in plot_axes]
    has_graphlet = observer_name == "triad_census" and dimension_name is not None
    if has_graphlet:
        add_triad_graphlet(
            plot_axes[0],
            dimension_name,
            below_legend=legend_flags[0],
        )

    if len(plot_axes) == 1:
        figure.subplots_adjust(
            left=0.09, right=0.79, bottom=0.15, top=0.90
        )
    else:
        figure.subplots_adjust(
            left=0.09, right=0.79, bottom=0.06, top=0.96, hspace=0.42
        )


def plot_agent_payoff_roles(observer, axis, *, show_runs=False):
    for dimension_name in AGENT_PAYOFF_PLOT_DIMENSIONS:
        color = AGENT_PAYOFF_ROLE_COLORS[dimension_name]
        plot_episode_series_observer(
            observer=observer,
            ax=axis,
            dimension_name=dimension_name,
            color=color,
            band_color=color,
            label=AGENT_PAYOFF_ROLE_LABELS[dimension_name],
            set_title=False,
            show_runs=show_runs,
        )


## Replay data


In [ ]:
data_root = project_root / "data/choi_2020/raw"
available_configs = list_choi_2020_configs(data_root)
available_configs[:5]

In [ ]:
config = available_configs[0]
max_steps = load_choi_2020_normalized_data(config).max_step
config, max_steps

## Observer settings


In [ ]:
action_printer = ActionPrintObserver(
    observation_sampling=ObservationSampling.periodic(1),
    print_empty_steps=False,
)
metric_observers_by_name = make_metric_observers(config.group_size)
metric_observers = tuple(metric_observers_by_name.values())
state_history_observer = StateHistoryObserver(
    observation_sampling=ObservationSampling.periodic(1),
    keep_runs=False,
)

(
    action_printer.__class__.__name__,
    list(metric_observers_by_name),
    state_history_observer.__class__.__name__,
)

## Build the replay simulator


In [ ]:
simulator = build_replay_simulator(
    config=config,
    observers=(action_printer, *metric_observers, state_history_observer),
    max_steps=max_steps,
)

fixed_layout = build_circle_layout(simulator.n_agents)


## Run the replay


In [ ]:
simulator.run(n_episodes=1, reset_observers=False)

(
    simulator.n_completed_episodes,
    state_history_observer.steps[-1],
    {observer.__class__.__name__: observer.steps[-1] for observer in metric_observers},
)


In [ ]:
metric_observers[-1].values[:25]

## Directed graph after each recorded state


In [ ]:
plot_mask = state_history_observer.steps <= 10
plot_graph_history(
    steps=state_history_observer.steps[plot_mask],
    graphs=state_history_observer.graphs[plot_mask],
    layout=fixed_layout,
    figure_title="Directed graph over the replay",
    max_cols=4,
)
plt.show()


## Metrics across the replay


In [ ]:
def build_replay_metric_plot(selection):
    observer_name, dimension_name = selection
    observer = metric_observers_by_name[observer_name]
    with plt.ioff():
        figure, axis = plt.subplots(figsize=(12, 4))

    if observer_name == "agent_payoff":
        plot_agent_payoff_roles(observer, axis)
        axis.set_title(f"{observer.label} - group size {config.group_size}")
    else:
        plot_episode_series_observer(
            observer=observer,
            ax=axis,
            dimension_name=dimension_name,
            set_title=False,
        )
        set_observer_plot_title(
            axis, observer_name, observer, dimension_name
        )

    finalize_metric_figure(
        figure, axis, observer_name, dimension_name
    )
    return figure


replay_observer_control, replay_plot_display = display_observer_explorer(
    metric_observers_by_name,
    build_replay_metric_plot,
)

## Aggregate metrics across group sizes


In [ ]:
group_sizes = sorted(choi_parameters)
group_colors = ["#1f4e79", "#c84c09", "#3a7d44"]

max_steps = 364
aggregate_metrics_by_group_size = {}
for group_size in group_sizes:
    group_configs = tuple(
        replay_config
        for replay_config in available_configs
        if replay_config.group_size == group_size
    )
    aggregate_metric_observers_by_name = make_metric_observers(
        group_size, aggregate_observer_aggregations
    )
    run_choi_2020_replay_batch(
        configs=group_configs,
        observers=tuple(aggregate_metric_observers_by_name.values()),
        max_steps=max_steps,
    )
    aggregate_metrics_by_group_size[group_size] = aggregate_metric_observers_by_name

reference_observers_by_name = aggregate_metrics_by_group_size[group_sizes[0]]


def build_aggregate_metric_plot(selection):
    observer_name, dimension_name = selection
    reference_observer = reference_observers_by_name[observer_name]

    if observer_name == "agent_payoff":
        with plt.ioff():
            figure, axes = plt.subplots(
                len(group_sizes),
                1,
                figsize=(12, 4 * len(group_sizes)),
                squeeze=False,
                sharex=True,
            )
        for group_size, axis in zip(group_sizes, axes.ravel()):
            plot_agent_payoff_roles(
                aggregate_metrics_by_group_size[group_size][observer_name],
                axis,
                show_runs=True,
            )
            axis.set_title(f"{reference_observer.label} - group size {group_size}")
        plot_axes = axes
    else:
        with plt.ioff():
            figure, axis = plt.subplots(figsize=(12, 4))
        for group_size, color in zip(group_sizes, group_colors):
            observer = aggregate_metrics_by_group_size[group_size][observer_name]
            plot_episode_series_observer(
                observer=observer,
                ax=axis,
                dimension_name=dimension_name,
                color=color,
                band_color=color,
                label=f"group size {group_size}",
                set_title=False,
                show_runs=True,
            )
        set_observer_plot_title(
            axis, observer_name, reference_observer, dimension_name
        )
        plot_axes = axis

    finalize_metric_figure(
        figure, plot_axes, observer_name, dimension_name
    )
    return figure

In [ ]:
aggregate_observer_control, aggregate_plot_display = display_observer_explorer(
    reference_observers_by_name,
    build_aggregate_metric_plot,
)

### Raw timelines and sigmoid fits

In [ ]:
sigmoid_observer_names = (
    "central_point_of_dominance",
    "centered_mean_welfare",
)
max_raw_timelines_per_panel = 80


def cached_sigmoid_parameters(observer_name, group_size):
    references = get_sigmoid_fit_reference_values(
        f"{observer_name}.mean",
        n_agents=int(group_size),
    )
    try:
        return tuple(
            float(references[parameter_name]["choi2020"])
            for parameter_name in ("A", "k", "t0")
        )
    except KeyError:
        return None

figure, axes = plt.subplots(
    len(sigmoid_observer_names),
    len(group_sizes),
    figsize=(5 * len(group_sizes), 3.2 * len(sigmoid_observer_names)),
    squeeze=False,
    sharex=True,
)

for row, observer_name in enumerate(sigmoid_observer_names):
    reference_observer = aggregate_metrics_by_group_size[group_sizes[0]][observer_name]
    for col, (group_size, color) in enumerate(zip(group_sizes, group_colors)):
        observer = aggregate_metrics_by_group_size[group_size][observer_name]
        result = observer.get()
        t = np.asarray(result.steps, dtype=float)
        mean = np.asarray(result.outputs["mean"], dtype=float).reshape(-1)
        std = np.asarray(result.outputs["std"], dtype=float).reshape(-1)
        params = cached_sigmoid_parameters(observer_name, group_size)

        axis = axes[row, col]
        runs = result.outputs.get("runs", ())
        if runs:
            run_indices = np.linspace(
                0,
                len(runs) - 1,
                min(max_raw_timelines_per_panel, len(runs)),
                dtype=int,
            )
            for run_index in run_indices:
                raw = np.asarray(runs[run_index], dtype=float).reshape(-1)
                axis.plot(t, raw, color="0.6", alpha=0.14, linewidth=0.8)

        axis.fill_between(
            t,
            mean - std,
            mean + std,
            color=color,
            alpha=0.18,
            linewidth=0,
            label="mean ± SD",
        )

        if params is None:
            axis.plot(t, mean, color=color, linewidth=2.0, label="mean")
            axis.set_title(f"n={group_size}; missing cached fit")
        else:
            A, k, t0 = params
            plot_sigmoid_fit(
                mean,
                A,
                k,
                t0,
                t=t,
                ax=axis,
                data_label="mean",
                fit_label="cached sigmoid fit",
                data_color=color,
                fit_color="black",
                fit_linestyle="--",
                show_legend=False,
            )
            axis.set_title(f"n={group_size}; A={A:.3g}, k={k:.3g}, t0={t0:.3g}")
        if col == 0:
            axis.set_ylabel(reference_observer.label)
        if row == len(sigmoid_observer_names) - 1:
            axis.set_xlabel("step")
        if row == 0 and col == len(group_sizes) - 1:
            axis.legend(loc="best", title="solid: mean; dashed: fit; shaded: ± SD")

figure.suptitle("Raw Choi replay timelines with mean ± SD and cached zero-start sigmoid fits", y=1.02)
figure.tight_layout()
plt.show()